In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import os 
import matplotlib.pyplot as plt 
import pandas as pd 
import numpy as np 
import tensorflow as tf 
from tensorflow import keras
from tensorflow.keras import layers
from keras.layers import Dense, Conv2D, MaxPooling2D, Flatten,Dropout

In [ ]:
print("TensorFlow Version:", tf.__version__)
print("GPU:", tf.config.list_physical_devices('GPU'))

In [ ]:
import os

os.listdir("/kaggle/input/datasets/puneet6060")

In [ ]:
import os

dataset_path = "/kaggle/input/datasets/puneet6060/intel-image-classification"

print(os.listdir(dataset_path))

In [ ]:
train_path = dataset_path + "/seg_train/seg_train"
print(os.listdir(train_path))

In [ ]:
#Count the images in each class

for folder in sorted(os.listdir(train_path)):
    folder_path = os.path.join(train_path, folder)
    print(f"{folder}: {len(os.listdir(folder_path))} images")

In [ ]:
# display one random picture from each class 

import random 

classes = sorted(os.listdir(train_path))
plt.figure(figsize=(12,10))

for i,cls in enumerate(classes) : 
    img_file = random.choice(os.listdir(os.path.join(train_path,cls)))
    img_path = os.path.join(train_path,cls,img_file)

    img = plt.imread(img_path)
    plt.subplot(2,3,i+1)
    plt.imshow(img)
    plt.title(cls)
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    train_path,
    validation_split = 0.2,
    batch_size = 32,
    image_size = (256,256),
    seed = 42,
    subset = 'training'
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    train_path,
    validation_split = 0.2,
    batch_size = 32,
    image_size = (256,256),
    seed = 42,
    subset = 'validation'
)

In [ ]:

# Normalize
def process(image, label):
    image = tf.cast(image / 255., tf.float32)
    return image, label

train_ds = train_ds.map(process)
val_ds = val_ds.map(process)

In [ ]:
from keras import Sequential
model = Sequential()
model.add(Conv2D(32,kernel_size=(3,3),padding='valid',activation='relu',input_shape=(256,256,3)))
model.add(MaxPooling2D(pool_size=(2,2),strides=2,padding='valid'))

model.add(Conv2D(64,kernel_size=(3,3),padding='valid',activation='relu'))
model.add(MaxPooling2D(pool_size=(2,2),strides=2,padding='valid'))

model.add(Conv2D(128,kernel_size=(3,3),padding='valid',activation='relu'))
model.add(MaxPooling2D(pool_size=(2,2),strides=2,padding='valid'))

model.add(Flatten())

model.add(Dense(128,activation = 'relu'))
model.add(Dropout(0.5))

model.add(Dense(64,activation = 'relu'))
model.add(Dropout(0.4))

model.add(Dense(6,activation = 'softmax'))

In [ ]:
model.summary()

In [ ]:
model.compile(
    optimizer = 'adam',
    loss = 'sparse_categorical_crossentropy',
    metrics = ['accuracy']
)

In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20
)